# ClearPath-ML: Data Exploration

This notebook provides an initial audit of the historical PM2.5 data fetched for Bengaluru. It focuses on identifying sensor locations, temporal patterns, and data quality issues.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

# Load data
df = pd.read_csv('../data/raw/bangalore_aqi_historical.csv')

print(f"Total records: {len(df)}")
df.head()

## 1. Outlier Audit
Analyzing the distribution of PM2.5 values to confirm the logic for removing readings > 500.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['value'], bins=100, kde=True)
plt.title('PM2.5 Distribution (Raw Data)')
plt.axvline(500, color='red', linestyle='--', label='Outlier Threshold (500)')
plt.legend()
plt.show()

outliers = df[df['value'] > 500]
print(f"Number of outliers (> 500): {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

## 2. Sensor Mapping
Identifying the most active sensors in the Bengalru region.

In [ ]:
sensor_counts = df.groupby('location_name')['value'].count().sort_values(ascending=False)
plt.figure(figsize=(10, 8))
sensor_counts.plot(kind='barh')
plt.title('Data points per Sensor Location')
plt.show()

## 3. Diurnal Patterns (Hourly Variation)
Visualizing how PM2.5 typically fluctuates throughout a 24-hour cycle.

In [ ]:
# Extract hour from the complex OpenAQ timestamp
def extract_local_hour(ts_str):
    ts_dict = ast.literal_eval(ts_str)
    local_dt = pd.to_datetime(ts_dict['local'])
    return local_dt.hour

df['hour'] = df['timestamp'].iloc[:1000].apply(extract_local_hour) # Testing on subset
sns.boxplot(x='hour', y='value', data=df[df['value'] <= 500].iloc[:10000])
plt.title('Hourly PM2.5 Trends (Bengaluru Sample)')
plt.show()